In [1]:
!pip install -q aiohttp

In [7]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Steam
!ls

Mounted at /content/drive
/content/drive/MyDrive/Steam
01_download.py		    emb_resnet18_pad.csv
02_audit.py		    emb_resnet18_pad.npy
03_extract_embeddings.py    emb_resnet18_squash.csv
04_diagnose.py		    emb_resnet18_squash.npy
05_linear_probe.py	    flagged_images.csv
06_metadata_baseline.py     games.csv
download_report.csv	    games_metadata.json
duplicate_images.csv	    image_audit.csv
emb_clip_plus_resnet.csv    images.tar
emb_clip_plus_resnet.npy    probe_grouped_emb_clip_plus_resnet.csv
emb_clip_plus_shuffled.csv  probe_grouped_emb_clip_squash.csv
emb_clip_plus_shuffled.npy  probe_grouped_emb_meta.csv
emb_clip_squash.csv	    probe_grouped_emb_meta_plus_clip.csv
emb_clip_squash.npy	    probe_per_label_emb_clip_plus_resnet.csv
emb_meta.csv		    probe_per_label_emb_clip_squash.csv
emb_meta.npy		    probe_per_label_emb_meta.csv
emb_meta_plus_clip.csv	    probe_per_label_emb_meta_plus_clip.csv
emb_meta_plus_clip.npy	    probe_results.csv
emb_meta_plus_clipshuf.csv  __pycache__
emb_

In [8]:
# 파일 추가 후 로딩 코드

%cd "/content/drive/MyDrive/Steam"
!ls -l 05_linear_probe.py

/content/drive/MyDrive/Steam
-rw------- 1 root root 19442 Aug 11 04:08 05_linear_probe.py


In [ ]:
# 세션 재시작 후 이미지 복구 코드

!tar -xf /content/drive/MyDrive/Steam/images.tar -C /content

## 1. 다운로드

In [3]:
!python 01_download.py --csv steam_image_links.csv --out /content/images \
    --concurrency 32

대상 50,870건 → /content/images/
     500/50,870   69.4 img/s  ETA  12.1분
   1,000/50,870   88.4 img/s  ETA   9.4분
   1,500/50,870   97.6 img/s  ETA   8.4분
   2,000/50,870  103.1 img/s  ETA   7.9분
   2,500/50,870  106.6 img/s  ETA   7.6분
   3,000/50,870  108.9 img/s  ETA   7.3분
   3,500/50,870  110.4 img/s  ETA   7.2분
   4,000/50,870  112.0 img/s  ETA   7.0분
   4,500/50,870  113.1 img/s  ETA   6.8분
   5,000/50,870  114.1 img/s  ETA   6.7분
   5,500/50,870  114.8 img/s  ETA   6.6분
   6,000/50,870  115.4 img/s  ETA   6.5분
   6,500/50,870  116.0 img/s  ETA   6.4분
   7,000/50,870  116.5 img/s  ETA   6.3분
   7,500/50,870  116.8 img/s  ETA   6.2분
   8,000/50,870  117.2 img/s  ETA   6.1분
   8,500/50,870  117.6 img/s  ETA   6.0분
   9,000/50,870  117.9 img/s  ETA   5.9분
   9,500/50,870  118.2 img/s  ETA   5.8분
  10,000/50,870  118.5 img/s  ETA   5.7분
  10,500/50,870  118.6 img/s  ETA   5.7분
  11,000/50,870  118.9 img/s  ETA   5.6분
  11,500/50,870  119.1 img/s  ETA   5.5분
  12,000/50,870  119.2 img/

## 2. 실사 점검 (손상/해상도/컬러모드/중복)

In [4]:
!python 02_audit.py --dir /content/images --out image_audit.csv

점검 대상 50,864개

  5,000/50,864
  10,000/50,864
  15,000/50,864
  20,000/50,864
  25,000/50,864
  30,000/50,864
  35,000/50,864
  40,000/50,864
  45,000/50,864
  50,000/50,864

=== 요약 ===
열림      50,864 / 50,864

--- 해상도 분포 (상위 10) ---
    460 x 215     50,774  (99.82%)
    460 x 210         23  ( 0.05%)
    460 x 181         13  ( 0.03%)
    444 x 208          8  ( 0.02%)
    416 x 215          7  ( 0.01%)
    460 x 214          5  ( 0.01%)
    450 x 215          4  ( 0.01%)
    459 x 215          4  ( 0.01%)
    292 x 136          3  ( 0.01%)
    329 x 153          2  ( 0.00%)
  ... 그 외 21종

--- 컬러 모드 ---
  RGB     50,465
  L          373
  CMYK        26

--- 파일 크기 ---
  평균   46.5 KB   중앙값   42.9 KB
  최소    1.4 KB   최대   7689.5 KB
  합계 2.42 GB

--- 완전 중복 이미지 ---
  중복 그룹 65개, 관련 게임 203개 (0.40%)
  가장 큰 그룹:
      37개 게임: [1443010, 1443011, 1443015, 1443016, 1443018, 1443019] ...
      10개 게임: [408100, 408101, 408102, 431190, 431191, 434940] ...
       9개 게임: [319000, 319010, 319011, 3190

## 3. 임베딩 추출 — 전처리 3종 비교용

In [5]:
for s in ['squash', 'crop', 'pad']:
    !python 03_extract_embeddings.py --encoder resnet18 --strategy $s --dir /content/images

감사 통과 50,864건
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100% 44.7M/44.7M [00:00<00:00, 142MB/s]
인코더 resnet18 (D=512, norm=imagenet) / 전처리 squash / device cuda

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Pl

## 4. CLIP

In [25]:
!python 03_extract_embeddings.py --encoder clip --strategy squash --dir /content/images

감사 통과 50,864건
Loading weights: 100% 398/398 [00:00<00:00, 20021.02it/s]
인코더 clip (D=512, norm=clip) / 전처리 squash / device cuda

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to a

## 5. Diagnose

In [23]:
!python 04_diagnose.py --dir /content/images

=== 1. 누락 ===
기대 50,870  확보 50,864  누락 6
  [1204870, 1398280, 1468960, 1611460, 2226640, 2381590]

  다운로드 리포트상 사유:
 app_id      status  http_status  bytes      error
1204870 bad_content          200    552 not a jpeg
1398280 bad_content          200    552 not a jpeg
1468960 bad_content          200    552 not a jpeg
1611460 bad_content          200    552 not a jpeg
2226640 bad_content          200    552 not a jpeg
2381590 bad_content          200    552 not a jpeg

  → 01_download.py를 그대로 다시 돌리면 이 6건만 재시도된다

=== 2. 크기 분포 ===
    0.1%       6.7 KB
    1.0%      12.1 KB
   50.0%      42.9 KB
   99.0%     140.8 KB
   99.9%     180.8 KB
  8.0 KB 미만: 104건
  1.0 MB 초과 : 2건

  큰 파일 상위:
       652960    7.87 MB  460x215
       604590    1.50 MB  460x215
  (해상도가 460x215인데 용량만 크면 노이즈가 많은 아트다. 리사이즈 후엔 문제되지 않는다)

=== 3. 저정보 이미지 판정 ===
  후보 104건을 실제로 열어 픽셀 분산 측정 중...
  픽셀 std 10.0 미만 (사실상 단색): 8건
 app_id  bytes  width  height  pixel_std  n_colors
1174390   1457    460     215   0.565043        1

## 6. 임베딩 품질 비교

In [28]:
!python 05_linear_probe.py \
    --emb emb_resnet18_squash emb_resnet18_crop emb_resnet18_pad emb_clip_squash \
    --top-tags 50

태그 라벨: 게임 49,022 / 태그 50종 (전체 441종, 빈도 200 이상 311종)
  최다: Indie, Singleplayer, Action, Adventure, Casual, Simulation, 2D, Strategy
  게임당 평균 양성 라벨 6.9개

device cuda / 라벨 tags(top50)

[emb_resnet18_squash]  N=49,016  D=512
  macro-AUC 0.5992

[emb_resnet18_crop]  N=49,016  D=512
  macro-AUC 0.5717

[emb_resnet18_pad]  N=49,016  D=512
  macro-AUC 0.5841

[emb_clip_squash]  N=49,016  D=512
  macro-AUC 0.6293

라벨: tags(top50)
          embedding     n  dim  macro_auc
    emb_clip_squash 49016  512     0.6293
emb_resnet18_squash 49016  512     0.5992
   emb_resnet18_pad 49016  512     0.5841
  emb_resnet18_crop 49016  512     0.5717

1위: emb_clip_squash
1~4위 격차 0.0576 — 유의미한 차이다. emb_clip_squash로 확정.

--- emb_clip_squash: 잘 맞히는 라벨 상위 10 ---
  Anime                        0.860
  Visual Novel                 0.781
  Action                       0.771
  Simulation                   0.753
  Casual                       0.745
  Horror                       0.744
  2D                           0.

결론
*   **전처리는 squash**
*   **인코더는 CLIP** (ResNet18 대비 +0.0301)

macro-AUC=0.63
*   이미지 단독으로 추천을 끌고 갈 수준은 아님
*   fusion에서 이미지의 기여가 작게 나올 가능성이 높음
*   오히려 어떤 태그를 이미지가 못 잡는지가 텍스트 모달리티가 담당해야 할 영역




## 7. Image Tower

In [ ]:
from image_tower import ImageTower, load_image_bank

bank, id2row = load_image_bank("emb_clip_squash", app_ids=games.app_id.values)
tower = ImageTower(in_dim=512, out_dim=64)      # out_dim은 텍스트 타워와 동일하게

rows   = torch.tensor([id2row[a] for a in batch_app_ids])
z_img  = tower(bank[rows])                       # (B, 64), L2 정규화됨
z_fuse = torch.cat([z_struct, z_text, z_img], dim=-1)



---



## Drive 백업

In [33]:
# 이미지는 tar 하나로 묶어서 옮긴다. 파일 5만 개를 개별 복사하면 수 시간 걸림
!tar -cf /content/images.tar -C /content images
!cp /content/images.tar {DRIVE}/

# 결과물은 개별 복사해도 금방
!cp emb_*.npy emb_*.csv image_audit.csv download_report.csv {DRIVE}/

cp: 'emb_clip_plus_resnet.npy' and '/content/drive/MyDrive/Steam/emb_clip_plus_resnet.npy' are the same file
cp: 'emb_clip_plus_shuffled.npy' and '/content/drive/MyDrive/Steam/emb_clip_plus_shuffled.npy' are the same file
cp: 'emb_clip_squash.npy' and '/content/drive/MyDrive/Steam/emb_clip_squash.npy' are the same file
cp: 'emb_resnet18_crop.npy' and '/content/drive/MyDrive/Steam/emb_resnet18_crop.npy' are the same file
cp: 'emb_resnet18_pad.npy' and '/content/drive/MyDrive/Steam/emb_resnet18_pad.npy' are the same file
cp: 'emb_resnet18_squash.npy' and '/content/drive/MyDrive/Steam/emb_resnet18_squash.npy' are the same file
cp: 'emb_clip_plus_resnet.csv' and '/content/drive/MyDrive/Steam/emb_clip_plus_resnet.csv' are the same file
cp: 'emb_clip_plus_shuffled.csv' and '/content/drive/MyDrive/Steam/emb_clip_plus_shuffled.csv' are the same file
cp: 'emb_clip_squash.csv' and '/content/drive/MyDrive/Steam/emb_clip_squash.csv' are the same file
cp: 'emb_resnet18_crop.csv' and '/content/drive



---


## CLIP+ResNet

In [29]:
import numpy as np, pandas as pd
a = np.load('emb_clip_squash.npy').astype(np.float32)
b = np.load('emb_resnet18_squash.npy').astype(np.float32)
ia = pd.read_csv('emb_clip_squash.csv').app_id.values
ib = pd.read_csv('emb_resnet18_squash.csv').app_id.values
assert (ia == ib).all(), "행 순서가 다르다"

c = np.concatenate([a, b], axis=1)          # 둘 다 이미 L2 정규화 상태
c /= np.linalg.norm(c, axis=1, keepdims=True)
np.save('emb_clip_plus_resnet.npy', c.astype(np.float16))
pd.DataFrame({'app_id': ia}).to_csv('emb_clip_plus_resnet.csv', index=False)
print(c.shape)

(50864, 1024)


In [30]:
!python 05_linear_probe.py --emb emb_clip_squash emb_clip_plus_resnet --top-tags 50

태그 라벨: 게임 49,022 / 태그 50종 (전체 441종, 빈도 200 이상 311종)
  최다: Indie, Singleplayer, Action, Adventure, Casual, Simulation, 2D, Strategy
  게임당 평균 양성 라벨 6.9개

device cuda / 라벨 tags(top50)

[emb_clip_squash]  N=49,016  D=512
  macro-AUC 0.6293

[emb_clip_plus_resnet]  N=49,016  D=1024
  macro-AUC 0.6777

라벨: tags(top50)
           embedding     n  dim  macro_auc
emb_clip_plus_resnet 49016 1024     0.6777
     emb_clip_squash 49016  512     0.6293

1위: emb_clip_plus_resnet
1~2위 격차 0.0484 — 유의미한 차이다. emb_clip_plus_resnet로 확정.

--- emb_clip_plus_resnet: 잘 맞히는 라벨 상위 10 ---
  Anime                        0.898
  Visual Novel                 0.833
  Horror                       0.820
  Cute                         0.799
  Action                       0.763
  Pixel Graphics               0.760
  Fantasy                      0.756
  2D                           0.752
  RPG                          0.751
  Simulation                   0.749

--- 못 맞히는 라벨 하위 10 ---
  Stylized                     0.610
 

In [31]:
import numpy as np, pandas as pd
a  = np.load('emb_clip_squash.npy').astype(np.float32)
b  = np.load('emb_resnet18_squash.npy').astype(np.float32)
ia = pd.read_csv('emb_clip_squash.csv').app_id.values

rng = np.random.default_rng(0)
c = np.concatenate([a, b[rng.permutation(len(b))]], axis=1)   # 짝만 깨뜨림
c /= np.linalg.norm(c, axis=1, keepdims=True)
np.save('emb_clip_plus_shuffled.npy', c.astype(np.float16))
pd.DataFrame({'app_id': ia}).to_csv('emb_clip_plus_shuffled.csv', index=False)

In [32]:
!python 05_linear_probe.py \
    --emb emb_clip_plus_resnet emb_clip_plus_shuffled emb_clip_squash --top-tags 50

태그 라벨: 게임 49,022 / 태그 50종 (전체 441종, 빈도 200 이상 311종)
  최다: Indie, Singleplayer, Action, Adventure, Casual, Simulation, 2D, Strategy
  게임당 평균 양성 라벨 6.9개

device cuda / 라벨 tags(top50)

[emb_clip_plus_resnet]  N=49,016  D=1024
  macro-AUC 0.6777

[emb_clip_plus_shuffled]  N=49,016  D=1024
  macro-AUC 0.6645

[emb_clip_squash]  N=49,016  D=512
  macro-AUC 0.6293

라벨: tags(top50)
             embedding     n  dim  macro_auc
  emb_clip_plus_resnet 49016 1024     0.6777
emb_clip_plus_shuffled 49016 1024     0.6645
       emb_clip_squash 49016  512     0.6293

1위: emb_clip_plus_resnet
1~3위 격차 0.0484 — 유의미한 차이다. emb_clip_plus_resnet로 확정.

--- emb_clip_plus_resnet: 잘 맞히는 라벨 상위 10 ---
  Anime                        0.898
  Visual Novel                 0.833
  Horror                       0.820
  Cute                         0.799
  Action                       0.763
  Pixel Graphics               0.760
  Fantasy                      0.756
  2D                           0.752
  RPG                 

In [35]:
!python 05_linear_probe.py \
    --emb emb_clip_squash emb_clip_plus_shuffled emb_clip_plus_resnet \
    --top-tags 50 --seeds 0 1 2 --pca-dim 512

태그 라벨: 게임 49,022 / 태그 50종 (전체 441종, 빈도 200 이상 311종)
  최다: Indie, Singleplayer, Action, Adventure, Casual, Simulation, 2D, Strategy
  게임당 평균 양성 라벨 6.9개

device cuda / 라벨 tags(top50)

[emb_clip_squash]  N=49,016  D=512
    seed 0: 0.7453
    seed 1: 0.7417
    seed 2: 0.7456
  macro-AUC 0.7442 ± 0.0022  (시드 3개)

[emb_clip_plus_shuffled]  N=49,016  D=1024 -> PCA 512
    seed 0: 0.7663
    seed 1: 0.7630
    seed 2: 0.7665
  macro-AUC 0.7652 ± 0.0020  (시드 3개)

[emb_clip_plus_resnet]  N=49,016  D=1024 -> PCA 512
    seed 0: 0.7688
    seed 1: 0.7653
    seed 2: 0.7675
  macro-AUC 0.7672 ± 0.0018  (시드 3개)

라벨: tags(top50)
             embedding     n  dim  macro_auc     sd
  emb_clip_plus_resnet 49016 1024     0.7672 0.0018
emb_clip_plus_shuffled 49016 1024     0.7652 0.0020
       emb_clip_squash 49016  512     0.7442 0.0022

1위: emb_clip_plus_resnet
1~3위 격차 0.0230 — 유의미한 차이다. emb_clip_plus_resnet로 확정.

--- emb_clip_plus_resnet: 잘 맞히는 라벨 상위 10 ---
  Anime                        0.908
  Visu

In [37]:
!python 05_linear_probe.py \
    --emb emb_clip_squash emb_clip_plus_resnet emb_clip_plus_shuffled \
          emb_resnet18_squash emb_resnet18_crop emb_resnet18_pad \
    --top-tags 50 --seeds 0 1 2 --pca-dim 400

태그 라벨: 게임 49,022 / 태그 50종 (전체 441종, 빈도 200 이상 311종)
  최다: Indie, Singleplayer, Action, Adventure, Casual, Simulation, 2D, Strategy
  게임당 평균 양성 라벨 6.9개

device cuda / 라벨 tags(top50)

[emb_clip_squash]  N=49,016  D=512 -> PCA 400
    seed 0: 0.7693
    seed 1: 0.7661
    seed 2: 0.7694
  macro-AUC 0.7683 ± 0.0019  (시드 3개)

[emb_clip_plus_resnet]  N=49,016  D=1024 -> PCA 400
    seed 0: 0.7690
    seed 1: 0.7654
    seed 2: 0.7677
  macro-AUC 0.7674 ± 0.0018  (시드 3개)

[emb_clip_plus_shuffled]  N=49,016  D=1024 -> PCA 400
    seed 0: 0.7663
    seed 1: 0.7631
    seed 2: 0.7666
  macro-AUC 0.7653 ± 0.0020  (시드 3개)

[emb_resnet18_squash]  N=49,016  D=512 -> PCA 400
    seed 0: 0.7069
    seed 1: 0.7052
    seed 2: 0.7047
  macro-AUC 0.7056 ± 0.0011  (시드 3개)

[emb_resnet18_crop]  N=49,016  D=512 -> PCA 400
    seed 0: 0.6894
    seed 1: 0.6894
    seed 2: 0.6883
  macro-AUC 0.6890 ± 0.0006  (시드 3개)

[emb_resnet18_pad]  N=49,016  D=512 -> PCA 400
    seed 0: 0.6929
    seed 1: 0.6921
    seed

ResNet18 concat 미채택

*   진짜와 순열 대조의 차이가 +0.0020
*   시드 SD가 0.0018~0.0022 -> 1 SD 안이라 0과 구분이 안 됨

결론: **CLIP ViT-B/32, 전처리 squash**

CLIP squash 0.7683 ± 0.0019 — 1위

CLIP+ResNet concat 0.7674

CLIP+섞은(대조군) 0.7653 — 이론대로 잡음을 더하면 손해

ResNet18: squash 0.7056 > pad 0.6920 > crop 0.6890

전처리 순위는 세 번의 서로 다른 조건에서 계속 squash > pad > crop



---
## 정형 데이터를 이미 가진 상태에서 이미지가 뭔가를 더하는가?


In [ ]:
!python 06_metadata_baseline.py --games games.csv --clip emb_clip_squash
!python 05_linear_probe.py --emb emb_meta_plus_clip emb_meta_plus_clipshuf \
    --top-tags 50 --seeds 0 1 2 --out probe_meta_vs_image.csv

결론: Yes! 정형 데이터를 이미 가진 상태에서도 이미지는 정보를 더함

정형 + 진짜 CLIP: 0.6961 ± 0.0010 / 정형 + 섞은 CLIP: 0.6678 ± 0.0011

차이 +0.0283, 표준편차의 약 28배